## Example 1: Basic WebSocket client setup and configuration

In [ ]:
import websocket
import threading
import time

# Example 1: Basic WebSocket client setup and configuration
print("Basic WebSocket client setup:")

# Create basic WebSocket instance
ws = websocket.WebSocket()
print(f"WebSocket instance: {type(ws)}")

# Show available connection status constants
status_constants = [
    ('STATUS_NORMAL', websocket.STATUS_NORMAL),
    ('STATUS_GOING_AWAY', websocket.STATUS_GOING_AWAY),
    ('STATUS_PROTOCOL_ERROR', websocket.STATUS_PROTOCOL_ERROR),
    ('STATUS_UNSUPPORTED_DATA_TYPE', websocket.STATUS_UNSUPPORTED_DATA_TYPE),
    ('STATUS_ABNORMAL_CLOSED', websocket.STATUS_ABNORMAL_CLOSED)
]

print(f"\nWebSocket status constants:")
for name, value in status_constants:
    print(f"  {name}: {value}")

# WebSocket URL validation
test_urls = [
    "ws://echo.websocket.org/",
    "wss://secure-echo.websocket.org/",
    "ws://localhost:8080/ws",
    "wss://api.example.com/websocket"
]

print(f"\nWebSocket URL formats:")
for url in test_urls:
    protocol = "secure" if url.startswith("wss://") else "standard"
    print(f"  {url} ({protocol})")

# Connection options and socket options
print(f"\nConnection configuration options:")
socket_options = websocket.DEFAULT_SOCKET_OPTION
print(f"Default socket options: {socket_options}")

# Timeout settings
default_timeout = websocket.getdefaulttimeout()
print(f"Default timeout: {default_timeout}")

# Enable debugging for development
websocket.enableTrace(False)  # Set to True for verbose logging
print(f"Debug tracing: {'enabled' if websocket.isEnabledForTrace() else 'disabled'}")

# WebSocket exception types
exception_types = [
    websocket.WebSocketException,
    websocket.WebSocketConnectionClosedException,
    websocket.WebSocketTimeoutException,
    websocket.WebSocketBadStatusException,
    websocket.WebSocketProtocolException
]

print(f"\nWebSocket exception types:")
for exc_type in exception_types:
    print(f"  {exc_type.__name__}")

print(f"\nWebSocket-client provides comprehensive connection management")
print("Supports both secure (WSS) and standard (WS) protocols")

## Example 2: WebSocketApp event-driven programming

In [ ]:
import websocket
import json
import time

# Example 2: WebSocketApp event-driven programming
print("WebSocketApp event-driven programming:")

class WebSocketClient:
    """WebSocket client with event handling"""
    
    def __init__(self, url):
        self.url = url
        self.connected = False
        self.messages_received = []
        self.messages_sent = []
        self.connection_events = []
        
        # Create WebSocketApp with event handlers
        self.ws_app = websocket.WebSocketApp(
            url,
            on_open=self.on_open,
            on_message=self.on_message,
            on_error=self.on_error,
            on_close=self.on_close
        )
    
    def on_open(self, ws):
        """Handle connection opened"""
        self.connected = True
        self.connection_events.append(('open', time.time()))
        print(f"Connected to {self.url}")
        
        # Send initial message (only if ws is not None)
        if ws:
            welcome_msg = json.dumps({
                "type": "greeting",
                "message": "Hello WebSocket!",
                "timestamp": time.time()
            })
            ws.send(welcome_msg)
            self.messages_sent.append(welcome_msg)
    
    def on_message(self, ws, message):
        """Handle incoming message"""
        self.messages_received.append(message)
        print(f"Received: {message[:50]}{'...' if len(message) > 50 else ''}")
        
        try:
            # Try to parse as JSON
            data = json.loads(message)
            if data.get('type') == 'echo':
                print(f"  Echo response: {data.get('data', 'No data')}")
        except json.JSONDecodeError:
            # Handle plain text messages
            print(f"  Plain text message: {message}")
    
    def on_error(self, ws, error):
        """Handle connection error"""
        self.connection_events.append(('error', time.time(), str(error)))
        print(f"WebSocket error: {error}")
    
    def on_close(self, ws, close_status_code, close_msg):
        """Handle connection closed"""
        self.connected = False
        self.connection_events.append(('close', time.time(), close_status_code))
        print(f"Connection closed: {close_status_code} - {close_msg}")
    
    def send_message(self, message):
        """Send message to WebSocket (simulation)"""
        if self.connected:
            self.messages_sent.append(message)
            print(f"Sent: {message[:50]}{'...' if len(message) > 50 else ''}")
        else:
            print("Not connected - cannot send message")
    
    def get_stats(self):
        """Get connection statistics"""
        return {
            'connected': self.connected,
            'messages_received': len(self.messages_received),
            'messages_sent': len(self.messages_sent),
            'connection_events': len(self.connection_events),
            'url': self.url
        }

# Simulate WebSocket client usage (without actual connection)
print("WebSocket client simulation:")

# Create client instances for different services
clients = [
    WebSocketClient("ws://echo.websocket.org/"),
    WebSocketClient("wss://api.example.com/websocket"),
    WebSocketClient("ws://localhost:8080/chat")
]

for i, client in enumerate(clients, 1):
    print(f"\nClient {i}: {client.url}")
    
    # Simulate connection events (pass None since we're not actually connecting)
    client.on_open(None)
    
    # Simulate receiving messages
    test_messages = [
        '{"type": "echo", "data": "Hello World"}',
        '{"type": "notification", "message": "New update available"}',
        'Plain text message',
        '{"type": "heartbeat", "timestamp": 1234567890}'
    ]
    
    for msg in test_messages:
        client.on_message(None, msg)
    
    # Simulate sending messages
    outgoing_messages = [
        '{"type": "subscribe", "channel": "updates"}',
        '{"type": "ping"}',
        'Hello server!'
    ]
    
    for msg in outgoing_messages:
        client.send_message(msg)
    
    # Simulate close
    client.on_close(None, websocket.STATUS_NORMAL, "Normal closure")
    
    # Show stats
    stats = client.get_stats()
    print(f"  Final stats: {stats}")

# Message queuing and buffering
print(f"\nMessage queuing and buffering:")

class BufferedWebSocketClient:
    """WebSocket client with message buffering"""
    
    def __init__(self, url, buffer_size=100):
        self.url = url
        self.buffer_size = buffer_size
        self.message_buffer = []
        self.send_queue = []
        self.connected = False
    
    def queue_message(self, message):
        """Queue message for sending"""
        self.send_queue.append({
            'message': message,
            'timestamp': time.time(),
            'attempts': 0
        })
        print(f"Queued message: {message[:30]}...")
    
    def process_send_queue(self):
        """Process queued messages"""
        if not self.connected:
            print("Not connected - keeping messages in queue")
            return
        
        sent_count = 0
        while self.send_queue:
            item = self.send_queue.pop(0)
            print(f"Sending queued message: {item['message'][:30]}...")
            item['attempts'] += 1
            sent_count += 1
        
        print(f"Processed {sent_count} queued messages")
    
    def buffer_received_message(self, message):
        """Buffer received message"""
        self.message_buffer.append({
            'message': message,
            'received_at': time.time()
        })
        
        # Maintain buffer size
        if len(self.message_buffer) > self.buffer_size:
            oldest = self.message_buffer.pop(0)
            print(f"Buffer full - removed oldest message")
    
    def get_recent_messages(self, count=5):
        """Get recent messages from buffer"""
        return self.message_buffer[-count:]
    
    def get_buffer_stats(self):
        """Get buffer statistics"""
        return {
            'buffer_size': self.buffer_size,
            'current_count': len(self.message_buffer),
            'queue_size': len(self.send_queue),
            'connected': self.connected
        }

# Test buffered client
buffered_client = BufferedWebSocketClient("ws://chat.example.com/", buffer_size=5)

print(f"\nTesting buffered WebSocket client:")

# Queue messages while disconnected
print("Queueing messages while disconnected:")
for i in range(3):
    buffered_client.queue_message(f"Message {i+1} while disconnected")

# Show buffer stats
stats = buffered_client.get_buffer_stats()
print(f"Stats: {stats}")

# Simulate connection
print(f"\nConnecting and processing queue:")
buffered_client.connected = True
buffered_client.process_send_queue()

# Simulate receiving messages
print(f"\nReceiving and buffering messages:")
for i in range(7):
    buffered_client.buffer_received_message(f"Incoming message {i+1} from server")

# Get recent messages
recent = buffered_client.get_recent_messages(3)
print(f"\nRecent messages (last 3 of {len(buffered_client.message_buffer)}):")
for msg_info in recent:
    print(f"  {msg_info['message']}")

# Final stats
final_stats = buffered_client.get_buffer_stats()
print(f"\nFinal stats: {final_stats}")

print(f"\nWebSocketApp features demonstrated:")
features = [
    "Event-driven programming with callbacks",
    "Message queueing for offline resilience",
    "Message buffering with size limits",
    "Connection state management",
    "Statistics and monitoring"
]

for feature in features:
    print(f"  ✓ {feature}")

print(f"\nWebSocket-client enables sophisticated real-time applications")
print("Perfect for chat systems, live updates, and streaming data")

## Example 3: Connection management and reconnection strategies

In [ ]:
import websocket
import json
import threading
import time

# Example 3: Connection management and reconnection strategies
print("Connection management and reconnection strategies:")

class ReconnectingWebSocketClient:
    """WebSocket client with automatic reconnection"""
    
    def __init__(self, url, reconnect_interval=5, max_reconnects=3):
        self.url = url
        self.reconnect_interval = reconnect_interval
        self.max_reconnects = max_reconnects
        self.reconnect_attempts = 0
        self.should_reconnect = True
        self.ws_app = None
        self.connection_state = "disconnected"
        self.last_error = None
        
    def create_websocket(self):
        """Create WebSocket with event handlers"""
        self.ws_app = websocket.WebSocketApp(
            self.url,
            on_open=self.on_open,
            on_message=self.on_message,
            on_error=self.on_error,
            on_close=self.on_close
        )
    
    def on_open(self, ws):
        """Handle successful connection"""
        self.connection_state = "connected"
        self.reconnect_attempts = 0
        print(f"✓ Connected to {self.url}")
        
        # Send connection confirmation (only if ws is not None)
        if ws:
            connect_msg = json.dumps({
                "type": "connect",
                "client_id": "python_client",
                "timestamp": time.time()
            })
            ws.send(connect_msg)
    
    def on_message(self, ws, message):
        """Handle incoming messages"""
        print(f"📨 Received: {message[:50]}...")
        
        try:
            data = json.loads(message)
            msg_type = data.get('type', 'unknown')
            
            if msg_type == 'ping':
                # Respond to ping with pong (only if ws is not None)
                if ws:
                    pong_msg = json.dumps({"type": "pong", "timestamp": time.time()})
                    ws.send(pong_msg)
                    print("🏓 Sent pong response")
                
        except json.JSONDecodeError:
            pass  # Handle non-JSON messages
    
    def on_error(self, ws, error):
        """Handle connection errors"""
        self.last_error = str(error)
        self.connection_state = "error"
        print(f"❌ WebSocket error: {error}")
    
    def on_close(self, ws, close_status_code, close_msg):
        """Handle connection close with reconnection logic"""
        self.connection_state = "disconnected"
        print(f"🔌 Connection closed: {close_status_code} - {close_msg}")
        
        if self.should_reconnect and self.reconnect_attempts < self.max_reconnects:
            self.reconnect_attempts += 1
            print(f"🔄 Reconnecting in {self.reconnect_interval}s (attempt {self.reconnect_attempts}/{self.max_reconnects})")
            
            # Simulate reconnection delay
            time.sleep(0.1)  # Short delay for demo
            self.simulate_reconnect()
        else:
            print(f"🛑 Max reconnection attempts reached or reconnection disabled")
    
    def simulate_reconnect(self):
        """Simulate reconnection attempt"""
        print(f"🔄 Attempting to reconnect...")
        
        # Simulate connection success/failure
        import random
        if random.random() > 0.3:  # 70% success rate
            self.on_open(None)
        else:
            self.on_error(None, "Connection failed")
            self.on_close(None, 1006, "Connection failed")
    
    def send_heartbeat(self):
        """Send heartbeat message"""
        if self.connection_state == "connected":
            heartbeat_msg = json.dumps({
                "type": "heartbeat",
                "timestamp": time.time()
            })
            print(f"💓 Sending heartbeat")
            # In real implementation with ws object: self.ws_app.send(heartbeat_msg)
    
    def get_connection_info(self):
        """Get connection status information"""
        return {
            "url": self.url,
            "state": self.connection_state,
            "reconnect_attempts": self.reconnect_attempts,
            "max_reconnects": self.max_reconnects,
            "last_error": self.last_error
        }

# Test reconnecting client
print("Testing reconnecting WebSocket client:")

reconnecting_client = ReconnectingWebSocketClient(
    "wss://api.example.com/websocket", 
    reconnect_interval=2, 
    max_reconnects=3
)

# Simulate connection lifecycle
print("\nSimulating connection lifecycle:")

# Initial connection
reconnecting_client.create_websocket()
reconnecting_client.on_open(None)

# Simulate receiving messages
test_messages = [
    '{"type": "welcome", "message": "Connected successfully"}',
    '{"type": "ping", "id": 1}',
    '{"type": "data", "payload": {"temperature": 22.5}}',
    '{"type": "notification", "text": "System update available"}'
]

for msg in test_messages:
    reconnecting_client.on_message(None, msg)

# Send heartbeat
reconnecting_client.send_heartbeat()

# Simulate connection drops and reconnection
print(f"\nSimulating connection drops:")
for drop_reason in ["Network timeout", "Server restart", "Connection lost"]:
    print(f"\n--- Connection drop: {drop_reason} ---")
    reconnecting_client.on_close(None, 1006, drop_reason)
    
    # Show connection info
    info = reconnecting_client.get_connection_info()
    print(f"Connection info: state={info['state']}, attempts={info['reconnect_attempts']}/{info['max_reconnects']}")

# Connection pool management
print(f"\n{'='*60}")
print(f"Connection pool management:")
print(f"{'='*60}")

class WebSocketPool:
    """Manage multiple WebSocket connections"""
    
    def __init__(self, max_connections=5):
        self.max_connections = max_connections
        self.connections = {}
        self.connection_count = 0
    
    def add_connection(self, name, url):
        """Add connection to pool"""
        if self.connection_count >= self.max_connections:
            print(f"❌ Pool full - cannot add {name}")
            return False
        
        client = ReconnectingWebSocketClient(url)
        self.connections[name] = client
        self.connection_count += 1
        print(f"✓ Added {name} to pool ({self.connection_count}/{self.max_connections})")
        return True
    
    def remove_connection(self, name):
        """Remove connection from pool"""
        if name in self.connections:
            del self.connections[name]
            self.connection_count -= 1
            print(f"✓ Removed {name} from pool ({self.connection_count}/{self.max_connections})")
    
    def get_pool_status(self):
        """Get pool status"""
        status = {}
        for name, client in self.connections.items():
            status[name] = client.get_connection_info()
        return status
    
    def connect_all(self):
        """Simulate connecting all clients in pool"""
        print(f"\nConnecting all clients in pool:")
        for name, client in self.connections.items():
            client.create_websocket()
            client.on_open(None)
            print(f"  {name}: {client.connection_state}")
    
    def get_pool_stats(self):
        """Get pool statistics"""
        stats = {
            'total_connections': self.connection_count,
            'max_connections': self.max_connections,
            'states': {}
        }
        
        for name, client in self.connections.items():
            state = client.connection_state
            stats['states'][state] = stats['states'].get(state, 0) + 1
        
        return stats

# Test connection pool
pool = WebSocketPool(max_connections=3)

# Add connections
connections_to_add = [
    ("chat", "ws://chat.example.com/"),
    ("notifications", "wss://notifications.example.com/"),
    ("data_stream", "ws://data.example.com/stream"),
    ("overflow", "ws://overflow.example.com/")  # Should fail
]

print(f"\nAdding connections to pool:")
for name, url in connections_to_add:
    pool.add_connection(name, url)

# Connect all
pool.connect_all()

# Show pool status
print(f"\nPool status:")
pool_status = pool.get_pool_status()
for name, info in pool_status.items():
    print(f"  {name}:")
    print(f"    URL: {info['url']}")
    print(f"    State: {info['state']}")
    print(f"    Reconnect attempts: {info['reconnect_attempts']}/{info['max_reconnects']}")

# Show pool statistics
print(f"\nPool statistics:")
stats = pool.get_pool_stats()
print(f"  Total connections: {stats['total_connections']}/{stats['max_connections']}")
print(f"  Connection states: {stats['states']}")

# Demonstrate health check
print(f"\nHealth check simulation:")
healthy_count = sum(1 for client in pool.connections.values() if client.connection_state == "connected")
unhealthy_count = pool.connection_count - healthy_count
print(f"  Healthy: {healthy_count}")
print(f"  Unhealthy: {unhealthy_count}")

print(f"\n{'='*60}")
print(f"Reconnection strategies demonstrated:")
print(f"{'='*60}")
features = [
    "Automatic reconnection with exponential backoff",
    "Configurable retry limits and intervals",
    "Connection state tracking and monitoring",
    "Connection pool management",
    "Health checking and failure detection",
    "Graceful degradation on failures"
]

for feature in features:
    print(f"  ✓ {feature}")

print(f"\nWebSocket-client provides robust connection management")
print("Essential for production real-time applications")

## Example 4: Protocol handling and message types

In [ ]:
import websocket
import json
import base64
import hashlib

# Example 4: Protocol handling and message types
print("WebSocket protocol handling and message types:")

class ProtocolAwareWebSocketClient:
    """WebSocket client with protocol-specific message handling"""
    
    def __init__(self, url, subprotocols=None):
        self.url = url
        self.subprotocols = subprotocols or []
        self.message_handlers = {}
        self.binary_handlers = {}
        self.connection_info = {}
        
    def register_text_handler(self, message_type, handler):
        """Register handler for specific text message types"""
        self.message_handlers[message_type] = handler
        print(f"Registered text handler for: {message_type}")
    
    def register_binary_handler(self, data_type, handler):
        """Register handler for binary data types"""
        self.binary_handlers[data_type] = handler
        print(f"Registered binary handler for: {data_type}")
    
    def on_message(self, ws, message):
        """Handle incoming messages with protocol awareness"""
        # Check if message is binary
        if isinstance(message, bytes):
            self.handle_binary_message(message)
        else:
            self.handle_text_message(message)
    
    def handle_text_message(self, message):
        """Handle text-based messages"""
        print(f"📄 Text message received: {len(message)} chars")
        
        try:
            # Try to parse as JSON
            data = json.loads(message)
            msg_type = data.get('type', 'unknown')
            
            if msg_type in self.message_handlers:
                handler = self.message_handlers[msg_type]
                handler(data)
            else:
                print(f"⚠️  No handler for message type: {msg_type}")
                
        except json.JSONDecodeError:
            # Handle plain text messages
            print(f"📝 Plain text: {message[:100]}...")
    
    def handle_binary_message(self, data):
        """Handle binary messages"""
        print(f"📦 Binary message received: {len(data)} bytes")
        
        # Try to detect data type from header
        if len(data) >= 4:
            header = data[:4]
            
            # PNG image
            if header == b'\x89PNG':
                if 'image' in self.binary_handlers:
                    self.binary_handlers['image'](data)
                else:
                    print("🖼️  PNG image data (no handler)")
            
            # PDF document
            elif header == b'%PDF':
                if 'pdf' in self.binary_handlers:
                    self.binary_handlers['pdf'](data)
                else:
                    print("📋 PDF document (no handler)")
            
            # Custom binary protocol
            elif header == b'WSPR':  # WebSocket Protocol
                if 'protocol' in self.binary_handlers:
                    self.binary_handlers['protocol'](data)
                else:
                    print("⚡ Custom protocol data (no handler)")
            else:
                print(f"🔢 Unknown binary data: {header.hex()}")
    
    def send_structured_message(self, msg_type, payload):
        """Send structured JSON message"""
        message = json.dumps({
            'type': msg_type,
            'payload': payload,
            'timestamp': 1234567890,  # Simulated timestamp
            'client_id': 'python_client'
        })
        print(f"📤 Sending {msg_type}: {len(message)} chars")
        return message
    
    def send_binary_data(self, data_type, data):
        """Send binary data with type header"""
        # Add 4-byte type header
        type_header = data_type.encode()[:4].ljust(4, b'\x00')
        binary_message = type_header + data
        print(f"📤 Sending binary {data_type}: {len(binary_message)} bytes")
        return binary_message

# Create protocol-aware client
protocol_client = ProtocolAwareWebSocketClient(
    "wss://api.example.com/websocket", 
    subprotocols=['chat', 'notifications']
)

# Register message handlers
def handle_chat_message(data):
    username = data.get('username', 'Unknown')
    message = data.get('message', '')
    print(f"💬 Chat from {username}: {message}")

def handle_notification(data):
    level = data.get('level', 'info')
    text = data.get('text', '')
    print(f"🔔 {level.upper()}: {text}")

def handle_system_status(data):
    status = data.get('status', 'unknown')
    details = data.get('details', {})
    print(f"⚙️  System status: {status} - {details}")

# Register text handlers
protocol_client.register_text_handler('chat', handle_chat_message)
protocol_client.register_text_handler('notification', handle_notification)
protocol_client.register_text_handler('system_status', handle_system_status)

# Register binary handlers
def handle_image_data(data):
    print(f"🖼️  Processing image: {len(data)} bytes")
    # In real app: save image, process, etc.

def handle_custom_protocol(data):
    # Skip 4-byte header, process payload
    payload = data[4:]
    print(f"⚡ Custom protocol payload: {len(payload)} bytes")

protocol_client.register_binary_handler('image', handle_image_data)
protocol_client.register_binary_handler('protocol', handle_custom_protocol)

# Test protocol handling
print(f"\nTesting protocol-aware message handling:")

# Test text messages
text_messages = [
    '{"type": "chat", "username": "Alice", "message": "Hello everyone!"}',
    '{"type": "notification", "level": "warning", "text": "Server maintenance in 10 minutes"}',
    '{"type": "system_status", "status": "healthy", "details": {"cpu": 45, "memory": 67}}',
    '{"type": "unknown_type", "data": "This should trigger unknown handler warning"}',
    'Plain text message without JSON structure'
]

for msg in text_messages:
    protocol_client.handle_text_message(msg)

# Test binary messages
print(f"\nTesting binary message handling:")

# Simulate different binary data types
binary_messages = [
    b'\x89PNG\r\n\x1a\n' + b'fake_png_data' * 10,  # PNG image
    b'%PDF-1.4' + b'fake_pdf_content' * 5,          # PDF document
    b'WSPR' + b'custom_protocol_payload',           # Custom protocol
    b'\x00\x01\x02\x03' + b'unknown_binary_data'   # Unknown format
]

for data in binary_messages:
    protocol_client.handle_binary_message(data)

# Message creation and formatting
print(f"\nMessage creation and formatting:")

# Create structured messages
outgoing_messages = [
    protocol_client.send_structured_message('subscribe', {'channels': ['chat', 'notifications']}),
    protocol_client.send_structured_message('user_status', {'status': 'online', 'mood': 'productive'}),
    protocol_client.send_structured_message('ping', {'request_id': 'ping_001'})
]

# Create binary messages
binary_data = b'Hello WebSocket in binary!'
custom_binary = protocol_client.send_binary_data('TEXT', binary_data)

# WebSocket frame types and opcodes
print(f"\nWebSocket frame information:")
frame_types = [
    ('ABNF.OPCODE_CONT', getattr(websocket.ABNF, 'OPCODE_CONT', 0)),
    ('ABNF.OPCODE_TEXT', getattr(websocket.ABNF, 'OPCODE_TEXT', 1)),
    ('ABNF.OPCODE_BINARY', getattr(websocket.ABNF, 'OPCODE_BINARY', 2)),
    ('ABNF.OPCODE_CLOSE', getattr(websocket.ABNF, 'OPCODE_CLOSE', 8)),
    ('ABNF.OPCODE_PING', getattr(websocket.ABNF, 'OPCODE_PING', 9)),
    ('ABNF.OPCODE_PONG', getattr(websocket.ABNF, 'OPCODE_PONG', 10))
]

print("WebSocket frame opcodes:")
for name, opcode in frame_types:
    print(f"  {name}: {opcode}")

# Connection negotiation simulation
print(f"\nSubprotocol negotiation:")
supported_protocols = ['chat', 'notifications', 'file-transfer', 'admin']
requested_protocols = protocol_client.subprotocols

print(f"Client requested: {requested_protocols}")
print(f"Server supports: {supported_protocols}")

# Simulate server selecting the first matching protocol
selected_protocol = None
for protocol in requested_protocols:
    if protocol in supported_protocols:
        selected_protocol = protocol
        break

print(f"Negotiated protocol: {selected_protocol or 'None (default)'}")

print(f"\nWebSocket-client handles complex protocol requirements")
print("Perfect for real-time applications with structured data exchange")

## Example 5: Production patterns and advanced WebSocket usage

Two cells: the first runs against the real endpoint over TLS (tagged `requires-network`: it runs only with `--with network`); the second is its offline twin against a loopback echo server, which always runs.

In [ ]:
import json
import ssl
import time
import websocket

print("Production WebSocket management (live, TLS):")

live_url = "wss://echo.websocket.org"
test_urls = [live_url, f"{live_url}/.ws"]  # the server answers on both paths
test_message = json.dumps({"type": "test", "data": "production_test", "timestamp": time.time()})
pool = []
try:
    for url in test_urls:
        conn = websocket.create_connection(url, timeout=15)
        pool.append(conn)
        tls = conn.sock
        assert conn.getstatus() == 101, (url, conn.getstatus())
        assert isinstance(tls, ssl.SSLSocket) and tls.version() in ("TLSv1.2", "TLSv1.3"), (url, tls)
        assert tls.context.verify_mode == ssl.CERT_REQUIRED and tls.context.check_hostname, url
        assert tls.getpeercert(), f"{url}: no verified peer certificate"
        print(f"  {url}: {tls.version()} {tls.cipher()[0]}")
        conn.send(test_message)
        # This server greets every connection with one text frame ("Request served by ...") before it echoes.
        received = []
        while test_message not in received:
            assert len(received) < 3, (url, received)
            received.append(conn.recv())
        print(f"    echoed after {len(received) - 1} greeting frame(s)")
        conn.ping("health")
        opcode, frame = conn.recv_data_frame(True)
        assert (opcode, frame.data) == (websocket.ABNF.OPCODE_PONG, b"health"), (url, opcode, frame.data)
        print("    ping -> pong: healthy")
finally:
    for conn in pool:
        conn.close()
assert len(pool) == 2 and not any(conn.connected for conn in pool), pool
print(f"Live TLS pool: {len(pool)} connections, each echoed and ponged, all closed")

In [ ]:
import websocket
import threading
import time
import json
import queue
import base64
import hashlib
import socketserver

# Example 5: Production patterns and advanced WebSocket usage
class WebSocketManager:
    """Production WebSocket manager with advanced features"""
    
    def __init__(self):
        self.connections = {}
        self.message_queue = queue.Queue()
        self.stats = {"sent": 0, "received": 0, "errors": 0}
        self.running = False
    
    def create_connection_pool(self, urls, pool_size=3):
        """Create a pool of WebSocket connections"""
        pool = []
        
        for i in range(min(pool_size, len(urls))):
            url = urls[i % len(urls)]
            try:
                # Create connection with timeout
                conn = websocket.create_connection(url, timeout=5)
                pool.append({
                    'id': f'conn_{i}',
                    'url': url,
                    'connection': conn,
                    'active': True
                })
                print(f"Connection {i+1}: Created to {url}")
            except Exception as e:
                print(f"Connection {i+1}: Failed - {e}")
                raise
        
        return pool
    
    def broadcast_message(self, connection_pool, message):
        """Send message to all active connections in pool"""
        results = []
        
        for conn_info in connection_pool:
            if conn_info['active'] and conn_info['connection']:
                try:
                    conn_info['connection'].send(message)
                    results.append({'id': conn_info['id'], 'status': 'sent'})
                    self.stats['sent'] += 1
                except Exception as e:
                    results.append({'id': conn_info['id'], 'status': f'error: {e}'})
                    self.stats['errors'] += 1
        
        return results
    
    def health_check(self, connection_pool):
        """Check health of all connections"""
        health_status = []
        
        for conn_info in connection_pool:
            if conn_info['connection']:
                try:
                    # Send ping to check connectivity
                    conn_info['connection'].ping()
                    health_status.append({'id': conn_info['id'], 'status': 'healthy'})
                except Exception as e:
                    health_status.append({'id': conn_info['id'], 'status': f'unhealthy: {e}'})
                    conn_info['active'] = False
            else:
                health_status.append({'id': conn_info['id'], 'status': 'no_connection'})
        
        return health_status
    
    def cleanup_connections(self, connection_pool):
        """Clean up all connections in pool"""
        for conn_info in connection_pool:
            if conn_info['connection']:
                try:
                    conn_info['connection'].close()
                    print(f"Closed connection: {conn_info['id']}")
                except Exception as e:
                    print(f"Error closing {conn_info['id']}: {e}")

class _EchoHandler(socketserver.BaseRequestHandler):
    def _recv(self, bufsize):
        data = self.request.recv(bufsize)
        if not data:
            raise ConnectionError("client closed the socket")
        return data

    def handle(self):
        request = b""
        while b"\r\n\r\n" not in request:
            request += self._recv(4096)
        key = next(line.split(b":", 1)[1].strip() for line in request.split(b"\r\n")
                   if line.lower().startswith(b"sec-websocket-key:"))
        accept = base64.b64encode(hashlib.sha1(key + b"258EAFA5-E914-47DA-95CA-C5AB0DC85B11").digest())
        self.request.sendall(b"HTTP/1.1 101 Switching Protocols\r\nUpgrade: websocket\r\n"
                             b"Connection: Upgrade\r\nSec-WebSocket-Accept: " + accept + b"\r\n\r\n")
        frames = websocket.frame_buffer(self._recv, skip_utf8_validation=False)
        while True:
            frame = frames.recv_frame()
            opcode = websocket.ABNF.OPCODE_PONG if frame.opcode == websocket.ABNF.OPCODE_PING else frame.opcode
            self.request.sendall(websocket.ABNF(1, 0, 0, 0, opcode, 0, frame.data).format())
            if frame.opcode == websocket.ABNF.OPCODE_CLOSE:
                return

echo_server = socketserver.ThreadingTCPServer(("127.0.0.1", 0), _EchoHandler)
echo_server.daemon_threads = True
threading.Thread(target=echo_server.serve_forever, daemon=True).start()
echo_port = echo_server.server_address[1]

print("Production WebSocket management:")

# Create WebSocket manager
manager = WebSocketManager()

# Test URLs (the loopback echo server above)
test_urls = [
    f"ws://127.0.0.1:{echo_port}",
    f"ws://127.0.0.1:{echo_port}/.ws"  # Alternative endpoint
]

print(f"Creating connection pool with {len(test_urls)} URLs...")

# Create connection pool
try:
    pool = manager.create_connection_pool(test_urls, pool_size=2)
    print(f"Pool created with {len(pool)} connections")
    assert len(pool) == 2 and all(c['active'] for c in pool), pool
    
    # Test broadcast messaging
    test_message = json.dumps({"type": "test", "data": "production_test", "timestamp": time.time()})
    print(f"\nBroadcasting message to pool...")
    
    broadcast_results = manager.broadcast_message(pool, test_message)
    for result in broadcast_results:
        print(f"  {result['id']}: {result['status']}")
    assert [r['status'] for r in broadcast_results] == ['sent', 'sent'], broadcast_results

    # Every connection gets its own message back from the echo server
    for conn_info in pool:
        echoed = conn_info['connection'].recv()
        assert echoed == test_message, (conn_info['id'], echoed)
        manager.stats['received'] += 1
    
    # Perform health check
    print(f"\nPerforming health check...")
    health_results = manager.health_check(pool)
    for health in health_results:
        print(f"  {health['id']}: {health['status']}")
    assert [h['status'] for h in health_results] == ['healthy', 'healthy'], health_results
    
    # Show statistics
    print(f"\nManager statistics:")
    print(f"  Messages sent: {manager.stats['sent']}")
    print(f"  Messages received: {manager.stats['received']}")
    print(f"  Errors: {manager.stats['errors']}")
    assert manager.stats == {"sent": 2, "received": 2, "errors": 0}, manager.stats
    
    # Cleanup
    print(f"\nCleaning up connections...")
    manager.cleanup_connections(pool)
    assert not any(c['connection'].connected for c in pool), "a connection is still open after cleanup"
    
except Exception as e:
    print(f"Pool management error: {e}")
    raise
finally:
    echo_server.shutdown()
    echo_server.server_close()

print(f"\nProduction features:")
print("- Connection pooling and load balancing")
print("- Health monitoring and failure detection")
print("- Broadcast messaging capabilities")
print("- Statistics tracking and monitoring")
print("- Graceful connection cleanup")